<a href="https://colab.research.google.com/github/kyjhub/diabetes-prediction-brfss/blob/main/prediction_of_diabetes_patient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BRFSS 건강 지표 기반 당뇨 라벨 분류

2015년 설문에서 이미 기록된 라벨을 건강 지표로 분류하는 실험입니다.
[데이터셋](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset)의
50:50 CSV를 노트북과 같은 폴더에 둡니다. Colab에서는 작업 폴더에 직접 업로드합니다.
설치 방법은 [README](README.md)에 있습니다.

학습용 데이터로 파라미터를 탐색하고, 검증용 데이터로 조기 종료와 임계값을 정합니다.
테스트 데이터는 마지막 평가 셀에서만 모델에 입력합니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

SEED = 156
DATA_PATH = Path("diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
OUTPUT_DIR = Path("outputs")

## 1. 데이터 읽기와 분할

라벨 비율을 유지하면서 원본 표본을 학습 60%, 검증 20%, 테스트 20%로 나눕니다.
이전 실험과 같은 입력 열을 비교하기 위해 `AnyHealthcare`는 계속 제외합니다.
BMI에 IQR 기준을 적용해 행을 삭제하지 않으며, 트리 모델의 기준 실험에서는
표준화와 SMOTE를 적용하지 않습니다.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"{DATA_PATH.name} 파일을 작업 폴더에 넣어주세요. 다운로드 링크는 README에 있습니다.")

data = pd.read_csv(DATA_PATH)
y = data["Diabetes_binary"].astype(int)
X = data.drop(columns=["Diabetes_binary", "AnyHealthcare"])

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED
)
print(f"학습: {len(X_train):,}행 / 검증: {len(X_val):,}행 / 테스트: {len(X_test):,}행")

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
ax.boxplot(X_train["BMI"])
ax.set(xticks=[1], xticklabels=["BMI"], title="BMI distribution (training set)")
fig.savefig(OUTPUT_DIR / "boxplot.png", dpi=150)
plt.show()

## 2. 학습 세트에서 파라미터 탐색, 검증 세트에서 조기 종료

학습 세트에서만 층화 5겹 교차 검증으로 8개 조합의 평균 ROC AUC를 비교합니다.
탐색 단계에서는 최대 300회 부스팅을 수행하고 조기 종료를 사용하지 않습니다.
선택한 파라미터로 학습 세트에서 다시 학습하며, 이때 상한을 1,000회로 늘리고 검증 세트의 ROC AUC로 조기 종료 시점을 고릅니다.

In [ ]:
base_params = {
    "objective": "binary:logistic",
    "tree_method": "hist",
    "n_estimators": 300,
    "random_state": SEED,
    "n_jobs": 2,
}
param_grid = {
    "max_depth": [3, 5],
    "min_child_weight": [1, 5],
    "learning_rate": [0.05, 0.1],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
search = GridSearchCV(
    XGBClassifier(**base_params),
    param_grid,
    cv=cv,
    scoring="roc_auc",
    refit=False,
)
search.fit(X_train, y_train)
print(f"최적 파라미터: {search.best_params_}")
print(f"교차 검증 평균 ROC AUC: {search.best_score_:.4f}")

final_params = {**base_params, **search.best_params_, "n_estimators": 1000}
model = XGBClassifier(**final_params, eval_metric="auc", early_stopping_rounds=25)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
n_rounds = model.best_iteration + 1
print(f"선택한 부스팅 횟수: {n_rounds}")
if n_rounds == final_params["n_estimators"]:
    print("조기 종료가 발생하지 않았습니다. n_estimators 상한을 늘려 다시 확인하세요.")

## 3. 검증 세트에서 분류 임계값 선택

기존 실험의 0.25~0.65 후보를 비교해 F1이 가장 큰 임계값을 고릅니다.
F1이 같으면 정밀도가 높은 값을 우선하고, 정밀도도 같으면 더 낮은 임계값을 고릅니다.
F1은 이 실험에서 선택한 기준입니다. 실제 적용에서 어떤 오류를 더 줄여야 하는지는 별도로 정해야 합니다.

In [ ]:
def threshold_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
    }


thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
val_proba = model.predict_proba(X_val)[:, 1]
validation_results = pd.DataFrame(
    [
        {"threshold": threshold, **threshold_metrics(y_val, val_proba, threshold)}
        for threshold in thresholds
    ]
)
best = validation_results.sort_values(
    ["f1", "precision", "threshold"], ascending=[False, False, True]
).iloc[0]
selected_threshold = float(best["threshold"])
display(validation_results.round(4))
print(f"선택한 임계값: {selected_threshold:.2f}")

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.plot(validation_results["threshold"], validation_results["precision"], marker="o", label="Precision")
ax.plot(validation_results["threshold"], validation_results["recall"], marker="o", label="Recall")
ax.axvline(selected_threshold, color="gray", linestyle="--", label="Selected threshold")
ax.set(xlabel="Classification threshold", ylabel="Score", ylim=(0, 1), title="Validation set")
ax.legend()
fig.savefig(OUTPUT_DIR / "Recall_precision_value.png", dpi=150)
plt.show()

## 4. 테스트 세트에서 최종 평가

검증 단계에서 정한 임계값을 고정하고 정확도, 정밀도, 재현율, F1 및 혼동 행렬을 구합니다.
ROC AUC는 이 임계값을 적용하기 전의 점수로 계산합니다.
이 실험은 50:50으로 구성한 2015년 설문 표본 안에서의 라벨 분류입니다.
정밀도 등은 실제 인구에 그대로 적용할 수 없으며, 미래 발병이나 개인 진단을 평가한 결과도 아닙니다.

In [ ]:
test_proba = model.predict_proba(X_test)[:, 1]
test_predictions = (test_proba >= selected_threshold).astype(int)
test_metrics = threshold_metrics(y_test, test_proba, selected_threshold)
test_metrics["roc_auc"] = roc_auc_score(y_test, test_proba)
print(f"최종 테스트 평가 (임계값 {selected_threshold:.2f})")
display(pd.DataFrame([test_metrics], index=["test"]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, ax=axes[0], colorbar=False)
axes[0].set_title("Test confusion matrix")
RocCurveDisplay.from_predictions(y_test, test_proba, ax=axes[1], name="XGBoost", plot_chance_level=True)
axes[1].set(
    xlabel="False positive rate (1 − specificity)",
    ylabel="True positive rate (sensitivity)",
    title="Test ROC curve",
)
fig.savefig(OUTPUT_DIR / "ROC_AUC_curve.png", dpi=150)
plt.show()